In [ ]:
%matplotlib inline
import sys, os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt

sys.path.insert(0, '/Users/yotameviatar/vs_code')
warnings.filterwarnings('ignore')

from HR import hr_config as config
from HR import hr_processor as processor

plt.rcParams.update({'figure.dpi': 110})
print('Setup complete')

In [ ]:
# ── What to show ──────────────────────────────────────────────────────────────
SUBJECT  = 'AB22'          # subject ID
SESSION  = None            # 'eve', 'mor', or None for both

# ── Scipy filter (overlaid on raw; does NOT affect rejection/scoring) ─────────
APPLY_FILTER = False       # ← flip this to True/False to toggle
FILTER_TYPE  = 'lowpass'   # 'lowpass', 'highpass', 'bandpass'
CUTOFF       = [0.3]       # Hz — one value for lp/hp, two for bandpass e.g. [0.05, 0.3]
FILTER_ORDER = 4

# ── Layout ────────────────────────────────────────────────────────────────────
NCOLS     = 6
SAVE_FIGS = True           # also write PNGs to OUTPUT_DIR/raw_show/<subject>/

In [ ]:
# ── Load cache & run analysis ─────────────────────────────────────────────────
cache_path = os.path.join(config.OUTPUT_DIR, '_cache', 'hr_cache.pkl')
with open(cache_path, 'rb') as f:
    _cache = pickle.load(f)

raw = _cache.get('sessions', _cache.get('trials'))
trials_data, traces = processor.apply_analysis_params(raw, config)

if SUBJECT not in trials_data:
    raise ValueError(f"Subject '{SUBJECT}' not found. Available: {sorted(trials_data.keys())}")

sess_keys = ([SESSION] if SESSION else
             sorted(k for k in trials_data[SUBJECT] if trials_data[SUBJECT][k]))

def _make_ba(sfreq):
    nyq = sfreq / 2.0
    if FILTER_TYPE in ('lowpass', 'highpass'):
        return butter(FILTER_ORDER, CUTOFF[0] / nyq, btype=FILTER_TYPE)
    return butter(FILTER_ORDER, [CUTOFF[0] / nyq, CUTOFF[1] / nyq], btype='bandpass')

def _apply_ba(sig, ba):
    return filtfilt(ba[0], ba[1], sig.astype(float))

for sk in sess_keys:
    trials = trials_data[SUBJECT].get(sk, [])
    n_rej  = sum(t['rejected'] for t in trials)
    filter_state = f'  [filter: {FILTER_TYPE} {CUTOFF} Hz {"ON" if APPLY_FILTER else "OFF"}]'
    print(f"{SUBJECT} {config.SESSION_MAP.get(sk,{}).get('label', sk)}: "
          f"{len(trials)} trials, {n_rej} rejected{filter_state}")

---
## View 1 — Individual Trials per Session
Each subplot = one trial.  
**Blue shading** = baseline window · **Red shading** = score window · **Orange overlay** = artifact-interpolated samples  
Title colour: black = clean pass · orange = pass with artifact · grey = rejected

In [ ]:
label_name = {1: 'Neg', 2: 'Neu'}
label_col  = {1: config.NEG_COLOR, 2: config.NEU_COLOR}

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)
    sfreq = trials[0]['sfreq']
    ba    = _make_ba(sfreq) if APPLY_FILTER else None

    n     = len(trials)
    nrows = max(1, (n + NCOLS - 1) // NCOLS)
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.0, nrows * 2.4), squeeze=False)
    axes_flat = axes.flatten()

    for i, t in enumerate(trials):
        ax      = axes_flat[i]
        tw      = t['times_wide']
        ep      = t['epoch_wide'].astype(float)
        rej     = t['rejected']
        lbl     = t.get('label', 0)
        bm      = t.get('baseline_mean')
        sc      = t.get('score')
        rr      = t.get('rejection_reason', '')
        art     = t.get('artifact_mask')
        any_art = t.get('any_artifact', False)

        if rej:
            line_col = '#95A5A6';  title_col = 'dimgray'
        else:
            line_col  = label_col.get(lbl, '#95A5A6')
            title_col = 'darkorange' if any_art else 'black'

        # Raw staircase — always shown in grey
        ax.step(tw, ep, where='post', color='#BBBBBB', lw=0.8, zorder=2)

        if ba is not None:
            # Filter ON: coloured filtered curve overlaid on grey raw
            try:
                ax.plot(tw, _apply_ba(ep, ba), color=line_col, lw=1.2, zorder=4)
            except Exception:
                pass
        else:
            # Filter OFF: colour the staircase directly
            ax.step(tw, ep, where='post', color=line_col, lw=1.0, zorder=3)

        if art is not None and np.any(art):
            y0, y1 = ep.min() - 0.5, ep.max() + 0.5
            ax.fill_between(tw, y0, y1, where=art, color='darkorange', alpha=0.18, zorder=1)

        ax.axvspan(config.BASELINE_TMIN, config.BASELINE_TMAX, alpha=0.10, color='#3498DB', zorder=0)
        ax.axvspan(config.SCORE_TMIN,    config.SCORE_TMAX,    alpha=0.08, color='#E74C3C',  zorder=0)
        if bm is not None and not np.isnan(float(bm)):
            ax.axhline(float(bm), color='goldenrod', ls=':', lw=0.9, alpha=0.85)
        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.45)

        ax.set_xlim(tw[0], tw[-1])
        ax.tick_params(labelsize=5.5)
        ax.spines[['top', 'right']].set_visible(False)

        art_mark = '~' if any_art and not rej else ''
        if rej:
            title = f'T{i+1} REJ\n{rr[:22] or "?"}'
        else:
            sc_str = f'{sc:.1f}%' if sc is not None and not np.isnan(float(sc)) else '—'
            title  = f'T{i+1} {art_mark}{label_name.get(lbl, "?")} {sc_str}'
        ax.set_title(title, fontsize=6.5, color=title_col, pad=2)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    filter_note = (f'  |  scipy {FILTER_TYPE} {CUTOFF} Hz  [ON]' if ba
                   else f'  |  filter [{FILTER_TYPE} {CUTOFF} Hz]  OFF')
    fig.suptitle(f'{SUBJECT} — {sess_label} — Individual trials  (N={n}){filter_note}',
                 fontsize=11, fontweight='bold')

    fig.legend(handles=[
        mpatches.Patch(color='#3498DB', alpha=0.5, label='Baseline window'),
        mpatches.Patch(color='#E74C3C', alpha=0.4, label='Score window'),
        mpatches.Patch(color='darkorange', alpha=0.4, label='Artifact (interpolated)'),
        plt.Line2D([0],[0], color='goldenrod', ls=':', label='Baseline mean'),
        plt.Line2D([0],[0], color=config.NEG_COLOR, label='Negative'),
        plt.Line2D([0],[0], color=config.NEU_COLOR, label='Neutral'),
        plt.Line2D([0],[0], color='#95A5A6', label='Rejected'),
    ], fontsize=7, loc='lower center', ncol=7, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_trials.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()

---
## View 2 — Full Session Overview
Continuous BPM signal across the entire session.  
**Green lines** = startle onsets where trial passed · **Red lines** = rejected trials  
Y-axis clipped to the physiological range (outlier spikes excluded).

In [ ]:
for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    tr = traces.get(SUBJECT, {}).get(sess_key, {})
    sig = tr.get('signal')
    if sig is None:
        print(f'{sess_key}: no session trace in cache (old format) — skipping')
        continue

    sig    = sig.astype(float)
    sfreq  = tr['sfreq']
    t_axis = np.arange(len(sig)) / sfreq
    ba     = _make_ba(sfreq) if APPLY_FILTER else None

    y_lo = max(getattr(config, 'HR_MIN_BPM', 30)  - 10, np.nanpercentile(sig, 1))
    y_hi = min(getattr(config, 'HR_MAX_BPM', 200) + 10, np.nanpercentile(sig, 99))

    fig, ax = plt.subplots(figsize=(20, 3.5))
    ax.step(t_axis, sig, where='post', color='#BBBBBB', lw=0.8, zorder=2, label='raw BPM')
    if ba is not None:
        try:
            ax.plot(t_axis, _apply_ba(sig, ba), color='#2C3E50', lw=0.9, zorder=3, label=f'filtered ({FILTER_TYPE} {CUTOFF} Hz)')
        except Exception:
            pass

    startle_sec = tr['startle_samps_sec']
    n = min(len(startle_sec), len(trials))
    for idx in range(n):
        t_star = float(startle_sec[idx])
        col    = '#27AE60' if not trials[idx].get('rejected') else '#E74C3C'
        ax.axvline(t_star, color=col, lw=1.0, alpha=0.8, zorder=4)

    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel('Time in session (s)', fontsize=9)
    ax.set_ylabel('BPM', fontsize=9)
    n_rej = sum(1 for t in trials[:n] if t.get('rejected'))
    filter_note = f'  |  filter {FILTER_TYPE} {CUTOFF} Hz  {"ON" if ba else "OFF"}'
    ax.set_title(
        f'{SUBJECT} — {sess_label} — Full session  '
        f'(green=pass, red=reject  {n_rej}/{n} rejected  '
        f'Y {y_lo:.0f}–{y_hi:.0f} BPM){filter_note}',
        fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'raw_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_session.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()